In [6]:
#PART-B
# ========== SETUP (run this first) ==========
!pip install "transformers<5.0.0" torch faiss-cpu sentence-transformers -q

# Authenticate with HuggingFace
# Get your free token at: https://huggingface.co/settings/tokens
from huggingface_hub import login

HF_TOKEN = "##############################"
login(HF_TOKEN)

import torch
import numpy as np
import faiss
import textwrap
from transformers import pipeline

print('Setup complete!')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

Setup complete!
Device: CPU


In [2]:
# ========== PROVIDED: Document Corpus ==========
# A set of short ML-focused documents. Your chatbot will answer questions about these.
DOCUMENTS = [
    """The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Google Brain. It replaced recurrent neural networks with self-attention
    mechanisms, allowing the model to process all tokens in parallel rather than sequentially.
    This enabled training on much larger datasets and dramatically improved performance on
    sequence-to-sequence tasks like translation and summarisation.""",

    """BERT (Bidirectional Encoder Representations from Transformers) is an encoder-only
    Transformer introduced by Google in 2018. It is trained using Masked Language Modelling,
    where 15% of input tokens are randomly masked and the model must predict them using
    both left and right context simultaneously. BERT is primarily used for text classification,
    named entity recognition, and extractive question answering.""",

    """GPT (Generative Pre-trained Transformer) is a decoder-only Transformer developed by
    OpenAI. It is trained on next-token prediction: given a sequence of tokens, predict the
    next one. GPT models use causal attention, meaning each token can only attend to previous
    tokens. This makes them naturally suited for text generation tasks. GPT-4, released in
    2023, is estimated to have around 1 trillion parameters.""",

    """RLHF (Reinforcement Learning from Human Feedback) is the training technique used to
    align language models with human preferences. It has three stages: supervised fine-tuning
    on instruction-following examples, training a reward model on human comparisons of
    model outputs, and using PPO (Proximal Policy Optimisation) to fine-tune the language
    model to maximise the reward model's scores. ChatGPT and Claude were both trained with RLHF.""",

    """RAG (Retrieval-Augmented Generation) is a technique that combines a language model with
    an external knowledge retrieval system. Documents are chunked, embedded using a sentence
    encoder, and stored in a vector database. At inference time, the user's query is embedded,
    the most similar document chunks are retrieved, and these chunks are injected into the
    LLM's prompt as context. This allows the model to answer questions about documents it was
    never trained on, without the hallucination risk of relying solely on parametric memory.""",


    """Word2Vec is a family of models introduced by Mikolov et al. at Google in 2013 for
    learning dense word embeddings. The two architectures are Skip-gram (predicts context
    words given a centre word) and CBOW (predicts a centre word given its context words).
    Words with similar meanings cluster together in the embedding space, enabling vector
    arithmetic such as: king - man + woman ≈ queen. Word2Vec embeddings are static —
    each word has one fixed vector regardless of context.""",

    """Few-shot prompting is a technique where 2-5 examples of a task are included directly
    in the prompt, allowing the LLM to infer the task format without any weight updates.
    Chain-of-Thought (CoT) prompting extends this by including step-by-step reasoning in
    the examples, which dramatically improves performance on arithmetic, logical, and
    multi-step reasoning tasks. Simply appending 'Let's think step by step' to a prompt
    enables zero-shot CoT reasoning.""",

    """Vector databases store high-dimensional embeddings and support efficient approximate
    nearest-neighbour search. Given a query vector, they return the K stored vectors with
    the highest cosine similarity or dot product. Popular options include FAISS (Facebook
    AI Similarity Search, open-source), Pinecone (managed cloud service), Chroma (lightweight,
    local), and Weaviate (open-source with a GraphQL API). FAISS uses techniques like
    product quantisation and inverted file indices to search billions of vectors in milliseconds.""",
]

print(f'Corpus: {len(DOCUMENTS)} documents loaded.')

Corpus: 8 documents loaded.


In [7]:
# ========== PROVIDED: Models ==========
# Do NOT change these — they are fixed for the assignment.

print('Loading embedding model...')
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = pipeline(
    'feature-extraction',
    model=EMBED_MODEL,
    tokenizer=EMBED_MODEL,
    device=-1,   # -1 = CPU; change to 0 for GPU if available
)
EMBED_DIM = 384
print(f'Embedding model loaded. Output dim: {EMBED_DIM}')

print('Loading text generation model...')
generator = pipeline(
    'text2text-generation',
    model='google/flan-t5-base',
    device=-1,
)
print('Generation model loaded.')

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Generation model loaded.


In [10]:
def embed_text(text: str) -> np.ndarray:
    """
    Embed a single string into a dense vector.

    Args:
        text: input string
    Returns:
        numpy array of shape (EMBED_DIM,), dtype float32

    TODO:
      1. raw = embedder(text)             # returns a nested list: [[[f, f, f, ...], ...], ...]
      2. Convert raw[0] to a 2D numpy array of shape (num_tokens, EMBED_DIM)
      3. Mean-pool along axis 0 -> shape (EMBED_DIM,)
      4. Return as float32
    """
    # YOUR CODE HERE
    output = embedder(text, return_tensors="pt")
    # Mean-pool over token dimension to get a single vector
    import torch
    vec = torch.tensor(output[0]).mean(dim=0)
    return vec.numpy().astype(np.float32)

# ===== Sanity Check =====
vec = embed_text('Hello world')
assert vec is not None, 'embed_text() returned None'
assert isinstance(vec, np.ndarray), f'Expected np.ndarray, got {type(vec)}'
assert vec.shape == (EMBED_DIM,), f'Expected shape ({EMBED_DIM},), got {vec.shape}'
assert vec.dtype == np.float32, f'Expected float32, got {vec.dtype}'
print(f'PASS: embed_text() works! Shape: {vec.shape}')

PASS: embed_text() works! Shape: (384,)


/tmp/ipykernel_13198/378658382.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  vec = torch.tensor(output[0]).mean(dim=0)


In [15]:
def chunk_documents(
    documents: list[str],
    chunk_size: int = 200,
    overlap: int = 40,
) -> list[str]:
    """
    Split each document into overlapping character-level chunks.

    Args:
        documents: list of document strings
        chunk_size: target size of each chunk in characters
        overlap: number of characters to overlap between consecutive chunks

    Returns:
        flat list of chunk strings

    TODO:
      For each document:
        Use a sliding window of size chunk_size with step (chunk_size - overlap).
        i.e., start positions: 0, chunk_size-overlap, 2*(chunk_size-overlap), ...
        Each chunk is doc[i : i + chunk_size].
        Strip whitespace from each chunk and skip empty ones.
      Return the flat list of all chunks across all documents.
    """
    # YOUR CODE HERE
    all_chunks = []
    step = chunk_size - overlap

    for doc in documents:
        # If a document is shorter than chunk_size, we still want to process it
        if not doc.strip():
            continue

        for i in range(0, len(doc), step):
            chunk = doc[i : i + chunk_size].strip()

            if chunk:  # Skip empty chunks
                all_chunks.append(chunk)

    return all_chunks


# ===== Sanity Check =====
test_chunks = chunk_documents(['a' * 500], chunk_size=200, overlap=40)
assert test_chunks is not None, 'chunk_documents() returned None'
assert len(test_chunks) > 1, 'Should produce multiple chunks for a 500-char doc'
assert all(len(c) <= 200 for c in test_chunks), 'A chunk exceeded chunk_size'
print(f'PASS: chunk_documents() works! {len(test_chunks)} chunks from 500-char doc')

chunks = chunk_documents(DOCUMENTS)
print(f'Full corpus: {len(DOCUMENTS)} documents -> {len(chunks)} chunks')
print(f'Sample chunk: "{chunks[0][:120]}..."')

PASS: chunk_documents() works! 4 chunks from 500-char doc
Full corpus: 8 documents -> 27 chunks
Sample chunk: "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Googl..."


In [16]:
def build_index(chunks: list[str]) -> tuple[faiss.Index, np.ndarray]:
    """
    Embed all chunks and build a FAISS inner-product index.

    Args:
        chunks: list of text chunks
    Returns:
        (index, embeddings) where:
          - index is a faiss.IndexFlatIP ready for search
          - embeddings is a float32 array of shape (len(chunks), EMBED_DIM)

    TODO:
      1. Embed each chunk using embed_text() -> stack into shape (N, EMBED_DIM)
         Hint: np.array([embed_text(c) for c in chunks])
      2. L2-normalise the embeddings in-place:
         faiss.normalize_L2(embeddings)   # modifies in-place
         (after normalisation, inner product = cosine similarity)
      3. Create index = faiss.IndexFlatIP(EMBED_DIM)
      4. Add embeddings to the index: index.add(embeddings)
      5. Return (index, embeddings)
    """
    # YOUR CODE HERE
    embeddings = np.array([embed_text(c) for c in chunks])
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(embeddings)
    return index, embeddings


print('Building index (this takes ~30 seconds on CPU)...')
index, embeddings = build_index(chunks)

# ===== Sanity Check =====
assert index is not None, 'build_index() returned None'
assert index.ntotal == len(chunks), f'Expected {len(chunks)} vectors in index, got {index.ntotal}'
assert embeddings.shape == (len(chunks), EMBED_DIM)
print(f'PASS: Index built! {index.ntotal} vectors of dim {EMBED_DIM}')

Building index (this takes ~30 seconds on CPU)...


/tmp/ipykernel_13198/378658382.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  vec = torch.tensor(output[0]).mean(dim=0)


PASS: Index built! 27 vectors of dim 384


In [18]:
def retrieve(
    question: str,
    index: faiss.Index,
    chunks: list[str],
    k: int = 3,
) -> list[tuple[str, float]]:
    """
    Retrieve the top-k most relevant chunks for a question.

    Args:
        question: user's query string
        index: FAISS index containing chunk embeddings
        chunks: original list of chunk strings (same order as index)
        k: number of chunks to retrieve
    Returns:
        list of (chunk_text, similarity_score) tuples, sorted by score descending

    TODO:
      1. Embed the question using embed_text() -> shape (EMBED_DIM,)
      2. Reshape to (1, EMBED_DIM) and convert to float32
      3. L2-normalise: faiss.normalize_L2(query_vec)
      4. Search: scores, indices = index.search(query_vec, k)
         scores and indices are shape (1, k) — take [0] to get 1D arrays
      5. Return [(chunks[idx], score) for idx, score in zip(indices[0], scores[0])]
    """
    # YOUR CODE HERE
    # 1. Embed the question using embed_text() -> shape (EMBED_DIM,)
    query_vec = embed_text(question)

    # 2. Reshape to (1, EMBED_DIM) and convert to float32
    query_vec = np.reshape(query_vec, (1, EMBED_DIM)).astype(np.float32)

    # 3. L2-normalise the query vector in-place
    faiss.normalize_L2(query_vec)

    # 4. Search the FAISS index
    scores, indices = index.search(query_vec, k)

    # 5. Build and return the list of (chunk_text, similarity_score) tuples
    # scores[0] and indices[0] extract the 1D arrays from the (1, k) outputs
    results = [(chunks[idx], float(score)) for idx, score in zip(indices[0], scores[0])]

    return results

# ===== Sanity Check =====
test_results = retrieve('What is BERT?', index, chunks, k=3)
assert test_results is not None, 'retrieve() returned None'
assert len(test_results) == 3, f'Expected 3 results, got {len(test_results)}'
assert isinstance(test_results[0][0], str), 'First element of each tuple should be a string'
assert isinstance(test_results[0][1], float), 'Second element of each tuple should be a float'
print('PASS: retrieve() works!')
print('\nTop 3 chunks for "What is BERT?":')
for i, (chunk, score) in enumerate(test_results):
    print(f'\n[{i+1}] Score: {score:.4f}')
    print(textwrap.fill(chunk[:200], width=80))

PASS: retrieve() works!

Top 3 chunks for "What is BERT?":

[1] Score: 0.6343
BERT (Bidirectional Encoder Representations from Transformers) is an encoder-
only     Transformer introduced by Google in 2018. It is trained using Masked
Language Modelling,     where 15% of input to

[2] Score: 0.5801
age Modelling,     where 15% of input tokens are randomly masked and the model
must predict them using     both left and right context simultaneously. BERT is
primarily used for text classification,

[3] Score: 0.2323
isation) to fine-tune the language     model to maximise the reward model's
scores. ChatGPT and Claude were both trained with RLHF.


/tmp/ipykernel_13198/378658382.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  vec = torch.tensor(output[0]).mean(dim=0)


In [21]:
def build_prompt(question: str, context_chunks: list[tuple[str, float]]) -> str:
    """
    Build an LLM prompt from a question and retrieved context chunks.

    Args:
        question: user's question
        context_chunks: list of (chunk_text, score) tuples from retrieve()
    Returns:
        formatted prompt string

    TODO:
      Build a prompt with this structure:

      Answer the question using only the context below.
      If the answer is not in the context, say "I don't know".

      Context:
      [Source 1]:

      [Source 2]:

      [Source 3]:

      Question:
      Answer:

    Each source should be on its own line, separated by a blank line.
    """
    # YOUR CODE HERE
    prompt_lines = [
        "Answer the question using only the context below.",
        'If the answer is not in the context, say "I dont know".',
        "",
        "Context:"
    ]

    # Dynamically format and add each source text separated by a blank line
    for i, (chunk_text, score) in enumerate(context_chunks):
        prompt_lines.append(f"[Source {i+1}]: {chunk_text}")
        prompt_lines.append("")  # Blank line separator

    # Append the final question and answer placeholders
    prompt_lines.append(f"Question: {question}")
    prompt_lines.append("Answer:")

    # Join everything together into a single string
    return "\n".join(prompt_lines)


# ===== Sanity Check =====
test_chunks_for_prompt = [('Transformers were introduced in 2017.', 0.95),
                           ('BERT is an encoder model.', 0.80)]
prompt = build_prompt('When were Transformers introduced?', test_chunks_for_prompt)
assert prompt is not None, 'build_prompt() returned None'
assert 'Source 1' in prompt, 'Prompt should contain [Source 1]'
assert 'Source 2' in prompt, 'Prompt should contain [Source 2]'
assert 'When were Transformers introduced?' in prompt, 'Prompt must contain the question'
assert 'Answer:' in prompt, 'Prompt must end with Answer:'
print('PASS: build_prompt() works!')
print('\nSample prompt:')
print('-' * 60)
print(prompt)
print('-' * 60)

PASS: build_prompt() works!

Sample prompt:
------------------------------------------------------------
Answer the question using only the context below.
If the answer is not in the context, say "I dont know".

Context:
[Source 1]: Transformers were introduced in 2017.

[Source 2]: BERT is an encoder model.

Question: When were Transformers introduced?
Answer:
------------------------------------------------------------


In [27]:
def generate_answer(prompt: str, max_new_tokens: int = 150) -> str:
    """
    Generate an answer from the LLM given a RAG prompt.

    Args:
        prompt: the formatted RAG prompt from build_prompt()
        max_new_tokens: maximum tokens to generate
    Returns:
        the generated answer string (stripped)

    TODO:
      1. Call generator(prompt, max_new_tokens=max_new_tokens)
         generator returns a list of dicts; get result[0]['generated_text']
      2. Strip whitespace and return the answer string
    """
    # YOUR CODE HERE
    result = generator(prompt, max_new_tokens=max_new_tokens)
    return result[0]['generated_text'].strip()


# ===== Sanity Check =====
test_prompt = 'Answer in one sentence. What is 2 + 2? Answer:'
answer = generate_answer(test_prompt, max_new_tokens=20)
assert answer is not None, 'generate_answer() returned None'
assert isinstance(answer, str), f'Expected str, got {type(answer)}'
assert len(answer) > 0, 'Answer is empty'
print(f'PASS: generate_answer() works! Sample output: "{answer}"')

PASS: generate_answer() works! Sample output: "2 + 2"


In [28]:
def rag_answer(question: str, index: faiss.Index, chunks: list[str], k: int = 3) -> str:
    """
    Full RAG pipeline: question -> retrieve -> prompt -> generate -> answer.

    TODO:
      1. Call retrieve() to get the top-k chunks
      2. Call build_prompt() with the question and retrieved chunks
      3. Call generate_answer() with the prompt
      4. Return the answer string
    """
    # YOUR CODE HERE
    # 1. Call retrieve() to get the top-k chunks and their scores
    context_chunks = retrieve(question, index, chunks, k=k)

    # 2. Call build_prompt() with the question and retrieved chunks
    prompt = build_prompt(question, context_chunks)

    # 3. Call generate_answer() with the prompt
    answer = generate_answer(prompt)

    # 4. Return the answer string
    return answer


# ===== Sanity Check =====
test_questions = [
    'What is BERT and what is it used for?',
    'How does RLHF work?',
    'What is the difference between Word2Vec and Transformer embeddings?',
]

print('========== RAG PIPELINE TEST ==========')
for q in test_questions:
    answer = rag_answer(q, index, chunks)
    assert answer is not None and len(answer) > 0, f'Got empty answer for: {q}'
    print(f'\nQ: {q}')
    print(f'A: {textwrap.fill(answer, width=80)}')

print('\nPASS: Full RAG pipeline works!')

========== RAG PIPELINE TEST ==========


/tmp/ipykernel_13198/378658382.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  vec = torch.tensor(output[0]).mean(dim=0)



Q: What is BERT and what is it used for?
A: BERT is primarily used for text classification, [Source 2]: age Modelling, where
15% of input tokens are randomly masked and the model must predict them using
both left and right context simultaneously. BERT is primarily used for text
classification, [Source 3]: RAG (Retrieval-Augmented Generation) is a technique
that combines a language model with an external knowledge retrieval system.
Documents are chunked, embedded using a sentence encoder, and st

Q: How does RLHF work?
A: It has three stages: supervised fine-tuning on instruction-fol [Source 2]:
isation) to fine-tune the language model to maximise the reward model's scores

Q: What is the difference between Word2Vec and Transformer embeddings?
A: Word2Vec embeddings are static — each word has one fixed vector regardless of
context.

PASS: Full RAG pipeline works!


In [29]:
def run_chatbot(index: faiss.Index, chunks: list[str]) -> None:
    """
    Run an interactive console chatbot loop.

    TODO:
      Implement a loop that:
      1. Prints a welcome message
      2. Repeatedly:
         a. Prompts the user for input with: question = input('You: ').strip()
         b. If question is empty, continue (ask again)
         c. If question.lower() is 'quit' or 'exit', print a goodbye message and break
         d. Otherwise:
            - Print 'Bot: Thinking...' so the user knows it is working
            - Call rag_answer(question, index, chunks)
            - Print 'Bot: '
            - Print a blank line for readability

    Example interaction:
    ─────────────────────────────────────
    ML Chatbot (type 'quit' to exit)

    You: What is a Transformer?
    Bot: Thinking...
    Bot: The Transformer architecture was introduced in 2017...

    You: quit
    Bot: Goodbye!
    ─────────────────────────────────────
    """
    # YOUR CODE HERE
    # 1. Print the welcome message
    print("ML Chatbot (type 'quit' to exit)")
    print()  # Blank line for layout structure

    # 2. Repeatedly prompt the user
    while True:
        # a. Prompt the user for input
        question = input('You: ').strip()

        # b. If question is empty, continue
        if not question:
            continue

        # c. If question matches exit keywords, say goodbye and break
        if question.lower() in ['quit', 'exit']:
            print('Bot: Goodbye!')
            break

        # d. Process valid user questions
        print('Bot: Thinking...')
        answer = rag_answer(question, index, chunks)
        print(f'Bot: {answer}')
        print()


# Run the chatbot!
# (In Colab, this will create an interactive input box)
run_chatbot(index, chunks)


ML Chatbot (type 'quit' to exit)

Bot: Thinking...


/tmp/ipykernel_13198/378658382.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  vec = torch.tensor(output[0]).mean(dim=0)


Bot: In the 2017 paper 'Attention Is All You Need' by Vaswani et al. at Google Brain. It replaced recurrent neural networks with self-attention mechanism

You: quit
Bot: Goodbye!


PART-A

A1.Ans:- QKt represent the dot product between the the query and the key which tells how ell the query and key are alligned. The dot product likes between negative to positive infinity so to normalise it, it is paassed through the softmax function which brings down its range between 0-1. The product of the resultant matrix with V for all the columns gives the change in the embedding vector.

A2.Ans:- For the first we will use the encoder and for the second case we will use the decoder since for text completion we require a model which does not reley on the data of the sentence after the word to be predicted.

A3.Ans:- RLHF prefers human company rather than human assstance becz its more cost and time effective, and making the model learn becomes easier.

A4.Ans:- The compny should work with a RAG model over a fine Tuned one because:-

1) Such a large amount of data is to be fed to the model requires high computational power and gpu since its the company's own data its completely new for the model.

2) As the documents are updated every week it becomes a heavy task to fine tune it every week.